# 04.2 Names, Objects, and References

04.1 established that a name is a label. This notebook builds the full mental
model — the one you will use to reason about every bug involving shared data,
function arguments, and copying.

## Theory

### The three-part picture

```
    NAME              REFERENCE           OBJECT
    ────              ─────────           ──────
    scores    ──────────────────────>    [90, 85, 77]
                                          id: 4382910
                                          type: list
```

- The **object** holds the data. It lives in memory and has an identity.
- The **reference** is the arrow — the fact that a name points at an object.
- The **name** is just a key in a namespace dictionary.

Multiple names can point at one object. An object with no names pointing at it
becomes garbage (04.6).

### What "passing to a function" really does

This is where the model pays off. Python is often described as "pass by value"
or "pass by reference". **Neither term fits.** The accurate name is
**pass by assignment** (sometimes "pass by object reference").

When you call `f(x)`, Python performs an assignment:

```python
parameter = argument      # exactly the rules from 04.1
```

So the parameter becomes **another name for the same object**. Whether the
caller sees changes depends entirely on whether you *mutate* the object or
*rebind* the name.

<table>
<tr><th>Inside the function</th><th>Caller sees it?</th><th>Why</th></tr>
<tr><td><code>items.append(4)</code></td><td><b>Yes</b></td><td>Mutates the shared object</td></tr>
<tr><td><code>items = [9]</code></td><td><b>No</b></td><td>Rebinds the local name only</td></tr>
</table>

That single table explains the majority of "why did my list change?" questions.

### Reference counting

CPython tracks how many references point at each object. When the count reaches
zero, the memory is reclaimed immediately. `sys.getrefcount()` lets you watch
this, which makes the model concrete rather than theoretical.

In [ ]:
import sys

# Create one object and give it several names.
scores = [90, 85, 77]
backup = scores
also_scores = scores

# All three names refer to ONE object.
print("scores      id:", id(scores))
print("backup      id:", id(backup))
print("also_scores id:", id(also_scores))
print("")
print("All the same object?", scores is backup is also_scores)

# getrefcount reports how many references exist.
# It is always one higher than you expect, because the argument
# passed INTO getrefcount is itself a temporary reference.
count = sys.getrefcount(scores)
print("")
print("References to that list:", count - 1, "(excluding the temporary)")

In [ ]:
import sys

# Watch the reference count rise and fall.
data = ["a", "b"]

def report(label):
    """Print the current reference count, excluding the temporary."""
    print(label.ljust(34), sys.getrefcount(data) - 1)

report("after data = [...]")

# Each new name adds a reference.
alias_one = data
report("after alias_one = data")

alias_two = data
report("after alias_two = data")

# Putting it in a container adds one too.
container = [data]
report("after container = [data]")

# Removing a name drops the count.
del alias_one
report("after del alias_one")

del alias_two
report("after del alias_two")

del container
report("after del container")

print("")
print("When the count would reach zero, CPython frees the memory at once.")

## Pass by assignment, demonstrated

The two functions below differ by one line. That line decides whether the caller
sees the change.

In [ ]:
def mutate_the_object(items):
    """Modify the object the caller passed in."""
    print("      inside, received id:", id(items))
    # append MUTATES the existing object.
    items.append("added inside")
    print("      inside, after append:", items)


def rebind_the_name(items):
    """Point the local name at a new object."""
    print("      inside, received id:", id(items))
    # Assignment REBINDS the local name. The caller's object is untouched.
    items = ["completely new list"]
    print("      inside, after rebind:", items)
    print("      inside, new id:      ", id(items))


# Case 1: mutation.
original = ["first"]
print("MUTATION")
print("   before:", original, "id:", id(original))
mutate_the_object(original)
print("   after: ", original, "<- caller sees the change")

# Case 2: rebinding.
original = ["first"]
print("")
print("REBINDING")
print("   before:", original, "id:", id(original))
rebind_the_name(original)
print("   after: ", original, "<- caller sees nothing")

print("")
print("The parameter was the same object in both cases. The difference is")
print("whether the function mutated it or repointed its own local name.")

### The same rule with an immutable argument

Immutable objects cannot be mutated, so a function can *only* rebind. This is why
integers and strings always look like "pass by value" — the mutation option
simply does not exist.

In [ ]:
def try_to_change_number(value):
    """Attempt to modify an int."""
    print("      inside, received:", value, "id:", id(value))
    # This can only rebind - int has no in-place modification.
    value += 100
    print("      inside, after +=:", value, "id:", id(value))


def try_to_change_string(text):
    """Attempt to modify a string."""
    print("      inside, received:", repr(text))
    # Same story - strings are immutable.
    text += " modified"
    print("      inside, after +=:", repr(text))


number = 5
print("INT")
print("   before:", number)
try_to_change_number(number)
print("   after: ", number, "<- unchanged, as expected")

message = "hello"
print("")
print("STR")
print("   before:", repr(message))
try_to_change_string(message)
print("   after: ", repr(message), "<- unchanged")

print("")
print("Nothing special happened here. The rule is the same as for lists -")
print("the function rebound its local name. It just had no other option.")

## The practical consequences

Three real patterns that follow directly from pass-by-assignment.

In [ ]:
# CONSEQUENCE 1: a function that mutates its argument is a hidden side effect.

def add_tax_bad(prices):
    """Modify the caller's list in place - surprising behaviour."""
    for index in range(len(prices)):
        prices[index] = round(prices[index] * 1.18, 2)
    # Returns nothing. The caller's data changed silently.


def add_tax_good(prices):
    """Return a NEW list, leaving the input untouched."""
    return [round(price * 1.18, 2) for price in prices]


original_prices = [100.0, 250.0]

# The bad version destroys the original.
working_copy = list(original_prices)
add_tax_bad(working_copy)
print("bad version  - input after call:", working_copy)

# The good version leaves it alone.
result = add_tax_good(original_prices)
print("good version - input after call:", original_prices)
print("good version - returned:        ", result)

print("")
print("RULE: either mutate and return None, or return a new value and")
print("mutate nothing. Never do both, and say which in the docstring.")

In [ ]:
# CONSEQUENCE 2: the mutable default argument trap.
# The default is created ONCE, when the function is DEFINED.

def collect_broken(item, target=[]):
    """Append to a default list - the classic bug."""
    target.append(item)
    return target


print("BROKEN - the default list persists between calls:")
print("   call 1:", collect_broken("a"))
print("   call 2:", collect_broken("b"), "<- 'a' is still there")
print("   call 3:", collect_broken("c"))

# Proof: the list is stored on the function object itself.
print("")
print("   the default lives here:", collect_broken.__defaults__)


def collect_fixed(item, target=None):
    """Use None as the sentinel and build a fresh list each call."""
    # Create a new list only when the caller did not supply one.
    if target is None:
        target = []

    target.append(item)
    return target


print("")
print("FIXED - a new list on every call:")
print("   call 1:", collect_fixed("a"))
print("   call 2:", collect_fixed("b"))
print("   call 3:", collect_fixed("c"))
print("")
print("   the default is now:", collect_fixed.__defaults__)

In [ ]:
# CONSEQUENCE 3: nested containers share their inner objects.

inner = [1, 2]
outer = [inner, inner, inner]

print("outer:", outer)
print("all three entries the same object?",
      outer[0] is outer[1] is outer[2])

# Changing it once changes every apparent copy.
inner.append(99)
print("")
print("after inner.append(99):", outer)

# The same trap with list multiplication.
grid = [[0] * 3] * 3
print("")
print("grid = [[0] * 3] * 3 ->", grid)

# Setting one cell appears to set a whole column.
grid[0][0] = 1
print("after grid[0][0] = 1  ->", grid, "<- every row changed")

# The correct way: build each row separately.
safe_grid = [[0] * 3 for _ in range(3)]
safe_grid[0][0] = 1
print("")
print("built with a comprehension ->", safe_grid, "<- correct")

## Seeing references in the interpreter

`gc.get_referrers()` shows what currently points at an object. This makes the
arrows in the diagram visible.

In [ ]:
import gc

# An object referenced from several places.
shared = {"role": "config"}
holder_list = [shared]
holder_dict = {"settings": shared}

# get_referrers finds the containers pointing at it.
referrers = gc.get_referrers(shared)

print("Things currently referring to that dict:")
for referrer in referrers:
    kind = type(referrer).__name__
    # Keep the preview short - namespaces are large.
    preview = repr(referrer)
    if len(preview) > 60:
        preview = preview[:57] + "..."
    print("   ", kind.ljust(10), preview)

print("")
print("The list and the dict both hold a reference. So does this")
print("notebook's own namespace, which is why a module dict appears.")

## Equality versus identity

Two questions that look similar and are not:

- `==` asks **"do these have the same value?"**
- `is` asks **"are these the same object?"**

Getting these confused produces bugs that appear intermittent, because the answer
can depend on how Python happens to have cached objects.

In [ ]:
# Two separate lists with identical contents.
first_list = [1, 2, 3]
second_list = [1, 2, 3]

print("first == second:", first_list == second_list, "<- same value")
print("first is second:", first_list is second_list, "<- different objects")
print("   id(first): ", id(first_list))
print("   id(second):", id(second_list))

# Now two names for one object.
third_list = first_list
print("")
print("first is third:", first_list is third_list, "<- same object")

print("")
print("RULE OF THUMB:")
print("   use == for values")
print("   use is ONLY for None, True, False, and genuine identity checks")

# The correct way to test for None.
value = None
print("")
print("   value is None:", value is None, "<- correct")
print("   value == None:", value == None, "<- works, but fragile")
print("")
print("== can be overridden by a class; `is` cannot. That is why None")
print("checks always use `is`. See 04.3 and Chapter 27.")

## Takeaways

1. A **name** points to an **object** via a **reference**. Many names can point
   at one object.
2. Python passes arguments **by assignment** — the parameter becomes another name
   for the caller's object.
3. **Mutating** the object is visible to the caller; **rebinding** the name is
   not.
4. Immutable arguments only look different because mutation is impossible for
   them — the rule is unchanged.
5. A function should either mutate and return `None`, or return a new value and
   mutate nothing.
6. **Mutable default arguments** are created once, at definition time. Use
   `None` as the sentinel.
7. `[[0] * 3] * 3` repeats a **reference**, not the row — use a comprehension.
8. `==` compares values, `is` compares identity. Use `is` for `None`.

## Try it yourself

1. Write a function that takes a list and appends to it. Call it twice with the
   same list. Then change it to rebind instead — what differs?
2. Use `sys.getrefcount()` to watch a count rise as you add names and fall as you
   `del` them.
3. Write `def f(x=[])` and call it three times. Then fix it with `None`.
4. Build a 3x3 grid with `[[0] * 3] * 3` and set one cell. Explain the result.
5. Find two objects where `==` is `True` but `is` is `False`, and two where both
   are `True`.